# Project Title - Building a dataset for European Countries from 2018 to 2023

## Project purpose

## project plans

## Project Procedures

### Step 1: Install Dependencies

In [1]:
# import packages
import pandas as pd
import numpy as np
import matplotlib
import comtradeapicall
import wbgapi as wb
import networkx
import sqlalchemy
import geopandas as gpd
import pycountry
import requests
import pyarrow
import os
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time  # The RateLimiter adds a 1.2 second delay between calls to respect Nominatim's usage policy.

### Step 2: Build the Country Table

In [2]:
CENTRAL_ASIA_ISO3 = {"KAZ", "UZB", "TKM", "TJK", "KGZ", "ARM", "AZE", "GEO"}


def build_country_table(save_path="data/european_countries.csv"):
    # make a new data directory to save all the data
    os.makedirs("data", exist_ok=True)
    # filter by Europe
    iso3_list = list(wb.region.members("ECS"))
    # write data to rows
    rows = []
    for iso3 in iso3_list:
        try:
            info = wb.economy.get(iso3)
            rows.append(
                {
                    "iso3": iso3,
                    "country_name": info["value"],
                }
            )
        except Exception as e:
            print(f"  Skipping {iso3}: {e}")
    # convert rows to pd.dataframe and save to local csv
    df = pd.DataFrame(rows)
    df = df[~df["iso3"].isin(CENTRAL_ASIA_ISO3)]
    df.to_csv(save_path, index=False)

    return df

In [3]:
build_country_table()

,iso3,country_name
0,POL,Poland
1,ROU,Romania
2,NOR,Norway
3,LUX,Luxembourg
4,ALB,Albania
5,LIE,Liechtenstein
6,PRT,Portugal
7,MNE,Montenegro
9,CZE,Czechia
10,UKR,Ukraine


### Step 3: Add Capital Coordinates
add capitals as the point on the map representing the country

In [4]:
def add_capital_coordinates(df):
    geolocator = Nominatim(user_agent="europe_trade_network")
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1.2)

    # get capital names from RestCountries API
    resp = requests.get(
        "https://restcountries.com/v3.1/region/europe?fields=cca3,capital"
    )
    capitals_map = {}
    for c in resp.json():
        iso3 = c.get("cca3")
        capital = c.get("capital", [None])[0]
        if iso3 and capital:
            capitals_map[iso3] = capital

    # Geocode each capital
    rows = []
    for _, row in df.iterrows():
        iso3 = row["iso3"]
        capital = capitals_map.get(iso3, None)

        if not capital:
            rows.append({"iso3": iso3, "capital": None, "lat": None, "lon": None})
            continue

        try:
            location = geocode(f"{capital}, Europe")
            if location:
                rows.append(
                    {
                        "iso3": iso3,
                        "capital": capital,
                        "lat": location.latitude,
                        "lon": location.longitude,
                    }
                )
            else:
                rows.append(
                    {"iso3": iso3, "capital": capital, "lat": None, "lon": None}
                )
        except Exception as e:
            rows.append({"iso3": iso3, "capital": capital, "lat": None, "lon": None})
    # turn all the capital rows into pd.dataframe
    coords_df = pd.DataFrame(rows)
    # merge to get a complete dataset of geographical location for visualization later
    merged = df.merge(coords_df, on="iso3", how="left")
    return merged

In [5]:
# check dataset and save to local csv
df = pd.read_csv("data/european_countries.csv")
df = add_capital_coordinates(df)
df.to_csv("data/european_countries.csv", index=False)
df

,iso3,country_name,capital,lat,lon
0,POL,Poland,Warsaw,52.232633,20.984259
1,ROU,Romania,Bucharest,44.461655,26.118277
2,NOR,Norway,Oslo,59.961451,10.925014
3,LUX,Luxembourg,Luxembourg,47.424383,0.697451
4,ALB,Albania,Tirana,41.330190,19.832111
5,LIE,Liechtenstein,Vaduz,NaN,NaN
6,PRT,Portugal,Lisbon,38.707629,-9.165507
7,MNE,Montenegro,Podgorica,NaN,NaN
8,CZE,Czechia,Prague,48.517489,-2.743307
9,UKR,Ukraine,Kyiv,50.452224,30.527536


In [6]:
df["lat"].notna().sum()

np.int64(39)

In [7]:
df = pd.read_csv("data/european_countries.csv")

# Show missing countries
missing = df[df["lat"].isna()][["iso3", "country_name", "capital"]]
print(f"Missing coordinates for {len(missing)} countries:")
print(missing.to_string(index=False))

Missing coordinates for 11 countries:
iso3    country_name            capital
 LIE   Liechtenstein              Vaduz
 MNE      Montenegro          Podgorica
 EST         Estonia            Tallinn
 GRL       Greenland                NaN
 FRO   Faroe Islands           Tórshavn
 TUR         Turkiye                NaN
 ISL         Iceland          Reykjavik
 SMR      San Marino City of San Marino
 CHI Channel Islands                NaN
 XKX          Kosovo                NaN
 IMN     Isle of Man            Douglas


> After investigation, there are two types of problems here:
> 1. No capital at all — CHI, GRL, TUR, XKX for which RestCountries didn't return one
> 2. Geocoding failed due to special characters (Tórshavn), unusual names (City of San Marino), or small territories
> 
> For easy fix, we do a manual patch dictionary with domain knowledge.

In [8]:
# create a manual patch with domain knowledge
MANUAL_PATCH = {
    "CHI": {"capital": "Saint Helier", "lat": 49.1880, "lon": -2.1010},
    "IMN": {"capital": "Douglas", "lat": 54.1524, "lon": -4.4861},
    "LIE": {"capital": "Vaduz", "lat": 47.1415, "lon": 9.5215},
    "MNE": {"capital": "Podgorica", "lat": 42.4304, "lon": 19.2594},
    "GRL": {"capital": "Nuuk", "lat": 64.1835, "lon": -51.7216},
    "ISL": {"capital": "Reykjavik", "lat": 64.1355, "lon": -21.8954},
    "TUR": {"capital": "Ankara", "lat": 39.9334, "lon": 32.8597},
    "SMR": {"capital": "City of San Marino", "lat": 43.9361, "lon": 12.4463},
    "FRO": {"capital": "Torshavn", "lat": 62.0107, "lon": -6.7741},
    "EST": {"capital": "Tallinn", "lat": 59.4370, "lon": 24.7536},
    "XKX": {"capital": "Pristina", "lat": 42.6629, "lon": 21.1655},
}
# add to local csv
df = pd.read_csv("data/european_countries.csv")

for iso3, patch in MANUAL_PATCH.items():
    mask = df["iso3"] == iso3
    df.loc[mask, "capital"] = patch["capital"]
    df.loc[mask, "lat"] = patch["lat"]
    df.loc[mask, "lon"] = patch["lon"]

df.to_csv("data/european_countries.csv", index=False)
df

,iso3,country_name,capital,lat,lon
0,POL,Poland,Warsaw,52.232633,20.984259
1,ROU,Romania,Bucharest,44.461655,26.118277
2,NOR,Norway,Oslo,59.961451,10.925014
3,LUX,Luxembourg,Luxembourg,47.424383,0.697451
4,ALB,Albania,Tirana,41.330190,19.832111
5,LIE,Liechtenstein,Vaduz,47.141500,9.521500
6,PRT,Portugal,Lisbon,38.707629,-9.165507
7,MNE,Montenegro,Podgorica,42.430400,19.259400
8,CZE,Czechia,Prague,48.517489,-2.743307
9,UKR,Ukraine,Kyiv,50.452224,30.527536


In [9]:
df["lat"].isna().sum()

np.int64(0)

### Step 4: Add Comtrade Numeric Codes
map each country's ISO3 code to the numeric code that UN Comtrade uses

In [10]:
def add_comtrade_codes(df):
    rows = []
    for _, row in df.iterrows():
        iso3 = row["iso3"]
        try:
            # get comtrade codes from pycountry
            country = pycountry.countries.get(alpha_3=iso3)
            if country:
                rows.append({"iso3": iso3, "comtrade_num": str(int(country.numeric))})
            else:
                rows.append({"iso3": iso3, "comtrade_num": None})
                print(f"No pycountry match for {iso3} ({row['country_name']})")
        except Exception as e:
            rows.append({"iso3": iso3, "comtrade_num": None})
            print(f"{iso3}: {e}")

    codes_df = pd.DataFrame(rows)
    return df.merge(codes_df, on="iso3", how="left")

In [11]:
df = pd.read_csv("data/european_countries.csv")
df = add_comtrade_codes(df)
df.to_csv("data/european_countries.csv", index=False)

No pycountry match for CHI (Channel Islands)
No pycountry match for XKX (Kosovo)


> XKX and CHI do not have non-standard ISO codes. Add manual patch with domain knowledge

In [12]:
# add comtrade patch for CHI and XKX
COMTRADE_CODE_PATCH = {
    "CHI": "831",  # Channel Islands Comtrade numeric code
    "XKX": "926",  # Kosovo Comtrade numeric code
}

df = pd.read_csv("data/european_countries.csv")

for iso3, code in COMTRADE_CODE_PATCH.items():
    df.loc[df["iso3"] == iso3, "comtrade_num"] = code

df.to_csv("data/european_countries.csv", index=False)

/var/folders/cv/pb7bdsd97cq7qt_b65khbfvr0000gn/T/ipykernel_17352/1207565778.py:10: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '831' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df["iso3"] == iso3, "comtrade_num"] = code


In [13]:
df["comtrade_num"].notna().sum()

np.int64(50)

## Step 5: Fetch World Bank Node Attributes

- Time span chosen to be the recent 5 years (2019 - 2024)
- 4 World Bank indicators are chosen
    - GDP: a country's economic size
    - population: scale
    - trade % of GDP: trade openness
    - FDI: financial attractiveness

In [14]:
# selected indicators
WB_INDICATORS = {
    "NY.GDP.MKTP.CD": "gdp_usd",
    "SP.POP.TOTL": "population",
    "NE.TRD.GNFS.ZS": "trade_pct_gdp",
    "BX.KLT.DINV.WD.GD.ZS": "fdi_inflow_pct_gdp",
}
# recent 5 years
YEARS = list(range(2019, 2024))
# add new data to local csv
df_countries = pd.read_csv("data/european_countries.csv")
iso3_list = df_countries["iso3"].tolist()
# extract data from wbgapi
frames = []
for code, col_name in WB_INDICATORS.items():
    try:
        df = wb.data.DataFrame(code, economy=iso3_list, time=YEARS, skipBlanks=False)
        df = df.reset_index().melt(
            id_vars="economy", var_name="year", value_name=col_name
        )
        df["year"] = df["year"].astype(str).str.extract(r"(\d{4})").astype(int)
        df = df.rename(columns={"economy": "iso3"})
        frames.append(df)
    except Exception as e:
        print(f"fail error: {e}")
# save to local csv
nodes = frames[0]
for f in frames[1:]:
    nodes = nodes.merge(f, on=["iso3", "year"], how="outer")

nodes.head(2)

,iso3,year,gdp_usd,population,trade_pct_gdp,fdi_inflow_pct_gdp
0,ALB,2019,1.558511e+10,2567801.0,75.382129,7.706215
1,ALB,2020,1.524146e+10,2528480.0,59.520699,7.018652


In [15]:
nodes.to_csv("data/nodes_raw.csv", index=False)

In [16]:
# check nodes: suppose to have 250 rows for 50 countries over 5 years.
nodes.shape

(250, 6)

## Step 6: Clean Node Table
clean the node table by:
- merge datasets
- handle missing values

In [ ]:
nodes_raw = pd.read_csv("data/nodes_raw.csv")
countries = pd.read_csv("data/european_countries.csv")

nodes = nodes_raw.merge(
    countries[["iso3", "country_name", "capital", "lat", "lon"]], on="iso3", how="left"
)

cols = ["gdp_usd", "population", "trade_pct_gdp", "fdi_inflow_pct_gdp"]

na_per_indicator = nodes[cols].isna().sum()
print("missing values per indicator")
print(na_per_indicator)
print("taken percentage")
print(na_per_indicator / len(nodes))

missing values per indicator
gdp_usd                6
population             0
trade_pct_gdp         30
fdi_inflow_pct_gdp    37
dtype: int64
taken percentage
gdp_usd               0.024
population            0.000
trade_pct_gdp         0.120
fdi_inflow_pct_gdp    0.148
dtype: float64


In [ ]:
print("Countries with at least missing value")
missing_mask = nodes[cols].isna().any(axis=1)
print(
    nodes[missing_mask][["iso3", "country_name", "year"] + cols].to_string(index=False)
)

Countries with at least missing value
iso3    country_name  year      gdp_usd  population  trade_pct_gdp  fdi_inflow_pct_gdp
 AND         Andorra  2019 3.155150e+09     76474.0            NaN           13.381807
 AND         Andorra  2020 2.891002e+09     77380.0            NaN            9.432285
 AND         Andorra  2021 3.324648e+09     78364.0            NaN           14.582656
 AND         Andorra  2022 3.380612e+09     79705.0            NaN           17.570235
 AND         Andorra  2023 3.785066e+09     80856.0            NaN            7.285975
 CHI Channel Islands  2019 1.003186e+10    165639.0            NaN                 NaN
 CHI Channel Islands  2020 9.439811e+09    166235.0            NaN                 NaN
 CHI Channel Islands  2021 1.115754e+10    166741.0            NaN                 NaN
 CHI Channel Islands  2022 1.130827e+10    167215.0            NaN                 NaN
 CHI Channel Islands  2023 1.250732e+10    167691.0            NaN                 NaN
 FRO 

> All missing values are systematically from contries or regions that may not be able to report data, we decided to remove those row.

In [ ]:
cols = ["gdp_usd", "population", "trade_pct_gdp", "fdi_inflow_pct_gdp"]
missing_countries = (
    nodes_raw[nodes_raw[cols].isna().any(axis=1)]
    .groupby("iso3")[cols]
    .apply(lambda x: x.isna().sum())
    .reset_index()
)
print(missing_countries.to_string(index=False))

iso3  gdp_usd  population  trade_pct_gdp  fdi_inflow_pct_gdp
 AND        0           0              5                   0
 CHI        0           0              5                   5
 FRO        0           0              0                   5
 GIB        5           0              5                   5
 GRL        0           0              0                   5
 IMN        1           0              5                   5
 LIE        0           0              5                   5
 MCO        0           0              5                   5
 SMR        0           0              0                   2


In [ ]:
CANDIDATE_INDICATORS = {
    "NY.GDP.MKTP.CD": "gdp_usd",
    "NY.GDP.PCAP.CD": "gdp_per_capita",
    "NY.GDP.MKTP.KD.ZG": "gdp_growth",
    "NE.EXP.GNFS.ZS": "exports_pct_gdp",
    "NE.IMP.GNFS.ZS": "imports_pct_gdp",
    "BN.CAB.XOKA.GD.ZS": "current_account_pct",
    "TX.VAL.MRCH.CD.WT": "merchandise_exports_usd",
    "NV.IND.TOTL.ZS": "industry_pct_gdp",
    "NV.SRV.TOTL.ZS": "services_pct_gdp",
}

YEARS = list(range(2019, 2024))
countries = pd.read_csv("data/european_countries.csv")
iso3_list = countries["iso3"].tolist()

frames = []
for code, col_name in CANDIDATE_INDICATORS.items():
    try:
        df = wb.data.DataFrame(code, economy=iso3_list, time=YEARS, skipBlanks=False)
        df = df.reset_index().melt(
            id_vars="economy", var_name="year", value_name=col_name
        )
        df["year"] = df["year"].astype(str).str.extract(r"(\d{4})").astype(int)
        df = df.rename(columns={"economy": "iso3"})
        frames.append(df)
    except Exception as e:
        print(f"{col_name}: {e}")

merged = frames[0]
for f in frames[1:]:
    merged = merged.merge(f, on=["iso3", "year"], how="outer")

# Show missing value count per indicator
print("Missing values per indicator (out of 250 rows)")
cols = list(CANDIDATE_INDICATORS.values())
print(merged[cols].isna().sum().to_string())

Missing values per indicator (out of 250 rows)
gdp_usd                     6
gdp_per_capita              6
gdp_growth                 11
exports_pct_gdp            30
imports_pct_gdp            30
current_account_pct        35
merchandise_exports_usd    41
industry_pct_gdp            6
services_pct_gdp            6


In [ ]:
# check outliers